In [1]:
import torch
import torch.nn as nn 
import os
import pandas as pd
from tqdm import tqdm
import csv
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPProcessor
import faiss
import numpy as np
import pickle
import h5py
import lightgbm as lgb
import gc

/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class PriceDataset(Dataset):
    def __init__(self, img_path, image_paths, item_names, bullet_points, 
                 product_descriptions, values, units, processor, max_length=77):
        self.img_dir = img_path
        self.image_paths = image_paths
        self.item_names = item_names
        self.bullet_points = bullet_points
        self.product_descriptions = product_descriptions
        self.values = values
        self.units = units
        self.processor = processor
        self.max_length = max_length
    
    def __len__(self):
        return len(self.image_paths)
    
    def _concatenate_text(self, idx):
        """
        Concatenate all text fields into one string.
        Prioritizes most important information first (for token limit).
        """
        text_parts = []
        
        # Item Name (most important)
        if pd.notna(self.item_names[idx]) and str(self.item_names[idx]).strip():
            text_parts.append(str(self.item_names[idx]).strip())
        
        # Quantity (concise and important for pricing)
        if pd.notna(self.values[idx]) and pd.notna(self.units[idx]):
            text_parts.append(f"{self.values[idx]} {self.units[idx]}")
        
        # Bullet Points (key features)
        if pd.notna(self.bullet_points[idx]) and str(self.bullet_points[idx]).strip():
            text_parts.append(str(self.bullet_points[idx]).strip())
        
        # Product Description (detailed, might be truncated)
        if pd.notna(self.product_descriptions[idx]) and str(self.product_descriptions[idx]).strip():
            text_parts.append(str(self.product_descriptions[idx]).strip())
        
        # Join with separator that CLIP handles well
        full_text = ". ".join(text_parts)
        
        # Handle empty text case
        if not full_text.strip():
            full_text = "product"  # Fallback text
        
        return full_text
    
    def __getitem__(self, idx):
        """
        Returns a single sample with image and text features.
        """
        # Load image
        img_filename = f"{self.image_paths[idx]}.jpg"
        img_path = os.path.join(self.img_dir, img_filename)
        
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            # Return a blank image as fallback
            img = Image.new('RGB', (224, 224), color='white')
        
        # Concatenate text fields
        full_text = self._concatenate_text(idx)
        
        # Process with CLIP processor
        inputs = self.processor(
            text=full_text,
            images=img,
            return_tensors="pt",
            padding="max_length",  # Consistent padding
            truncation=True,
            max_length=self.max_length
        )
        
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0)
        }

In [3]:
class KNNFeatureAugmenter:
    def __init__(self, k=10, embedding_dim=512, use_gpu=False):
        """
        KNN-based feature augmentation for embeddings.
        
        Args:
            k: Number of nearest neighbors to consider
            embedding_dim: Dimension of the embeddings
            use_gpu: Whether to use GPU for FAISS operations (set to False for CPU-only)
        """
        self.k = k
        self.embedding_dim = embedding_dim
        self.use_gpu = use_gpu
        
        # Initialize FAISS indices for image and text embeddings
        self.img_index = None
        self.text_index = None
        
        # Store embeddings for meta-feature calculation (on CPU for FAISS)
        self.img_embeddings_cpu = None
        self.text_embeddings_cpu = None
        
        # Keep GPU versions for fast computation
        self.img_embeddings_gpu = None
        self.text_embeddings_gpu = None
        
    def _create_faiss_index(self, embeddings_cpu):
        """Create FAISS index for embeddings (CPU-only)."""
        # Ensure embeddings are on CPU and float32 for FAISS
        embeddings_np = embeddings_cpu.cpu().numpy().astype(np.float32)
        
        # Create CPU index (Inner product for cosine similarity with normalized vectors)
        index = faiss.IndexFlatIP(self.embedding_dim)
        
        # Add embeddings to index
        index.add(embeddings_np)
        
        return index
    
    def fit(self, img_embeddings, text_embeddings):
        """
        Build KNN indices from training embeddings.
        
        Args:
            img_embeddings: Image embeddings tensor [N, embedding_dim]
            text_embeddings: Text embeddings tensor [N, embedding_dim]
        """
        # Normalize embeddings for cosine similarity
        img_embeddings_norm = torch.nn.functional.normalize(img_embeddings, p=2, dim=1)
        text_embeddings_norm = torch.nn.functional.normalize(text_embeddings, p=2, dim=1)
        
        # Store CPU versions for FAISS
        self.img_embeddings_cpu = img_embeddings_norm.cpu()
        self.text_embeddings_cpu = text_embeddings_norm.cpu()
        
        # Store GPU versions for fast computation (if available)
        if img_embeddings.is_cuda:
            self.img_embeddings_gpu = img_embeddings_norm
            self.text_embeddings_gpu = text_embeddings_norm
        else:
            self.img_embeddings_gpu = self.img_embeddings_cpu
            self.text_embeddings_gpu = self.text_embeddings_cpu
        
        # Create FAISS indices (CPU-only)
        self.img_index = self._create_faiss_index(self.img_embeddings_cpu)
        self.text_index = self._create_faiss_index(self.text_embeddings_cpu)
        
    def _calculate_knn_features(self, query_embeddings, index, stored_embeddings_gpu):
        """
        Calculate KNN-based meta-features for query embeddings.
        
        Returns:
            knn_features: Tensor of meta-features [batch_size, 4]
                - mean_similarity: Average similarity to k nearest neighbors
                - std_similarity: Standard deviation of similarities
                - min_similarity: Minimum similarity to k nearest neighbors
                - diversity_score: Measure of diversity among neighbors
        """
        # Move query to CPU for FAISS search
        query_np = query_embeddings.cpu().numpy().astype(np.float32)
        
        # Search for k+1 neighbors (including self)
        similarities, indices = index.search(query_np, self.k + 1)
        
        # Remove self-similarity (first result is usually the query itself)
        similarities = similarities[:, 1:]  # [batch_size, k]
        indices = indices[:, 1:]
        
        # Convert similarities back to GPU tensors
        similarities = torch.tensor(similarities, device=query_embeddings.device, dtype=query_embeddings.dtype)
        
        # Calculate meta-features on GPU
        mean_similarity = similarities.mean(dim=1)
        std_similarity = similarities.std(dim=1)
        min_similarity = similarities.min(dim=1)[0]
        
        # Calculate diversity score (average pairwise distance among neighbors)
        diversity_scores = []
        for i in range(len(indices)):
            neighbor_indices = indices[i]
            # Get neighbor embeddings (use GPU version if available)
            neighbor_embeddings = stored_embeddings_gpu[neighbor_indices]  # [k, embedding_dim]
            
            # Calculate pairwise cosine distances on GPU
            if len(neighbor_embeddings) > 1:
                similarity_matrix = torch.mm(neighbor_embeddings, neighbor_embeddings.t())
                # Get upper triangular part (excluding diagonal)
                triu_indices = torch.triu_indices(len(neighbor_embeddings), len(neighbor_embeddings), offset=1, device=neighbor_embeddings.device)
                pairwise_similarities = similarity_matrix[triu_indices[0], triu_indices[1]]
                diversity_score = 1.0 - pairwise_similarities.mean()  # Higher diversity = lower similarity
            else:
                diversity_score = torch.tensor(0.0, device=query_embeddings.device, dtype=query_embeddings.dtype)
            
            diversity_scores.append(diversity_score)
        
        diversity_scores = torch.stack(diversity_scores)
        
        # Stack all features
        knn_features = torch.stack([
            mean_similarity,
            std_similarity,
            min_similarity,
            diversity_scores
        ], dim=1)
        
        return knn_features
    
    def transform(self, img_embeddings, text_embeddings):
        """
        Calculate KNN meta-features for given embeddings.
        
        Args:
            img_embeddings: Image embeddings tensor [batch_size, embedding_dim]
            text_embeddings: Text embeddings tensor [batch_size, embedding_dim]
            
        Returns:
            img_knn_features: Image KNN meta-features [batch_size, 4]
            text_knn_features: Text KNN meta-features [batch_size, 4]
        """
        if self.img_index is None or self.text_index is None:
            raise ValueError("Must call fit() before transform()")
        
        # Normalize query embeddings
        img_embeddings_norm = torch.nn.functional.normalize(img_embeddings, p=2, dim=1)
        text_embeddings_norm = torch.nn.functional.normalize(text_embeddings, p=2, dim=1)
        
        # Calculate KNN features (CPU search, GPU computation)
        img_knn_features = self._calculate_knn_features(
            img_embeddings_norm, self.img_index, self.img_embeddings_gpu
        )
        text_knn_features = self._calculate_knn_features(
            text_embeddings_norm, self.text_index, self.text_embeddings_gpu
        )
        
        return img_knn_features, text_knn_features

In [4]:
class CLIPPricePredictor(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32", k_neighbors=10):
        super().__init__()
        # Load CLIP model
        self.clip = CLIPModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="cuda"
        )
        
        # Freeze CLIP parameters
        for param in self.clip.parameters():
            param.requires_grad = False
        
        # Get embedding dimension from CLIP
        self.embedding_dim = self.clip.config.projection_dim
        
        # Initialize KNN augmenter (CPU-only for FAISS)
        self.knn_augmenter = KNNFeatureAugmenter(
            k=k_neighbors, 
            embedding_dim=self.embedding_dim, 
            use_gpu=False  # Set to False for CPU-only FAISS
        )
        
        # Flag to track if KNN is fitted
        self.knn_fitted = False
    
    @torch.no_grad()
    def extract_features(self, pixel_values, input_ids, attention_mask):
        """
        Extract frozen multimodal embeddings from CLIP.
        Combines image and text features in the shared embedding space.
        """
        self.clip.eval()
        
        # Get CLIP outputs
        outputs = self.clip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Extract normalized embeddings from CLIP's projection space
        image_embeds = outputs.image_embeds  # [batch_size, projection_dim]
        text_embeds = outputs.text_embeds    # [batch_size, projection_dim]
        
        return image_embeds, text_embeds
    
    def fit_knn(self, dataloader):
        """
        Fit KNN indices using all training data.
        
        Args:
            dataloader: DataLoader containing training samples
        """
        print("Fitting KNN indices...")
        all_img_embeds = []
        all_text_embeds = []
        
        self.eval()
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Extracting embeddings for KNN"):
                pixel_values = batch['pixel_values'].to(next(self.parameters()).device)
                input_ids = batch['input_ids'].to(next(self.parameters()).device)
                attention_mask = batch['attention_mask'].to(next(self.parameters()).device)
                
                img_embeds, text_embeds = self.extract_features(
                    pixel_values, input_ids, attention_mask
                )
                
                # Keep on GPU for now, will be handled in KNN augmenter
                all_img_embeds.append(img_embeds)
                all_text_embeds.append(text_embeds)
        
        # Concatenate all embeddings (keep on GPU)
        all_img_embeds = torch.cat(all_img_embeds, dim=0)
        all_text_embeds = torch.cat(all_text_embeds, dim=0)
        
        # Fit KNN augmenter (it will handle GPU-to-CPU movement internally)
        self.knn_augmenter.fit(all_img_embeds, all_text_embeds)
        self.knn_fitted = True
        print("KNN indices fitted successfully!")
    
    def forward(self, pixel_values, input_ids, attention_mask=None):
        """
        Forward pass that returns all four numerical features.
        
        Returns:
            dict with keys: 'text_embeds', 'img_embeds', 'text_knn', 'img_knn'
        """
        # Extract CLIP features
        with torch.no_grad():
            img_embeds, text_embeds = self.extract_features(
                pixel_values, input_ids, attention_mask
            )
        
        # Calculate KNN features if fitted
        if self.knn_fitted:
            img_knn_features, text_knn_features = self.knn_augmenter.transform(
                img_embeds, text_embeds
            )
        else:
            # Use zeros as placeholder if KNN not fitted
            batch_size = img_embeds.size(0)
            device = img_embeds.device
            img_knn_features = torch.zeros(batch_size, 4, device=device, dtype=img_embeds.dtype)
            text_knn_features = torch.zeros(batch_size, 4, device=device, dtype=text_embeds.dtype)
        
        return {
            'text_embeds': text_embeds,
            'img_embeds': img_embeds, 
            'text_knn': text_knn_features,
            'img_knn': img_knn_features
        }

In [5]:
def load_features_from_h5(file_path):
    features = {}
    with h5py.File(file_path, 'r') as f:
        features['text_embeds'] = f['text_embeds'][:]
        features['img_embeds'] = f['img_embeds'][:]
        features['text_knn'] = f['text_knn'][:]
        features['img_knn'] = f['img_knn'][:]
        if 'prices' in f:
            features['prices'] = f['prices'][:]
        if 'indices' in f:
            features['indices'] = f['indices'][:]
    return features

def prepare_features_for_lgbm(features_dict):
    X = np.concatenate([
        features_dict['text_embeds'], 
        features_dict['img_embeds'], 
        features_dict['text_knn'], 
        features_dict['img_knn']
    ], axis=1)
    return X

In [6]:
def extract_test_features_and_predict(test_csv_path, test_img_dirs, 
                                     trained_model_path, lgbm_model_path,
                                     output_csv_path, batch_size=32):
    
    # Load test data
    test_df = pd.read_csv(test_csv_path)
    print(f"Loaded test data: {len(test_df)} samples")
    
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)
    
    model = CLIPPricePredictor(k_neighbors=10)
    
    # Load LightGBM model
    print(f"Loading LightGBM model from {lgbm_model_path}")
    lgbm_model = lgb.Booster(model_file=lgbm_model_path)
    
    all_predictions = []
    all_sample_ids = []
    
    for part_idx, img_dir in enumerate(test_img_dirs):
        print(f"\nProcessing test part {part_idx + 1}/{len(test_img_dirs)}: {img_dir}")
        
        # Determine which samples belong to this part
        # This assumes test data is split across parts - adjust logic as needed
        part_start = part_idx * (len(test_df) // len(test_img_dirs))
        part_end = (part_idx + 1) * (len(test_df) // len(test_img_dirs))
        if part_idx == len(test_img_dirs) - 1:  # Last part gets remainder
            part_end = len(test_df)
        
        part_df = test_df.iloc[part_start:part_end].copy()
        
        if len(part_df) == 0:
            continue
            
        
        test_dataset = PriceDataset(
            img_path = img_dir,
            image_paths=part_df['sample_id'].values,
            item_names=part_df['Item Name'].values,
            bullet_points=part_df['Bullet Points'].values,
            product_descriptions=part_df['Product Description'].values,
            values=part_df['Value'].values,
            units=part_df['Unit'].values,
            processor=processor
        )
        
        test_dataloader = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )
        
        # Extract features for this part
        print(f"Extracting features for part {part_idx + 1}...")
        part_features = extract_features_from_dataloader(model, test_dataloader)
        
        # Prepare features for LightGBM
        X_test_part = prepare_features_for_lgbm(part_features)
        
        # Make predictions
        print(f"Making predictions for part {part_idx + 1}...")
        predictions_part = lgbm_model.predict(X_test_part)
        
        # Convert from log scale back to original scale
        predictions_part = np.exp(predictions_part)
        
        # Store results
        all_predictions.extend(predictions_part)
        all_sample_ids.extend(part_df['sample_id'].values)
        
        # Clean up memory
        del part_features, X_test_part, predictions_part, test_dataset, test_dataloader
        gc.collect()
        torch.cuda.empty_cache()
    
    # Create output DataFrame
    output_df = pd.DataFrame({
        'sample_id': all_sample_ids,
        'price': all_predictions
    })
    test_df
    final_predictions = test_df[['sample_id']].merge(
        output_df, 
        on='sample_id', 
        how='left'
    )
    final_predictions['sample_id'] = final_predictions['sample_id'].astype(int)
    final_predictions['price'] = final_predictions['price'].astype(float)
    final_predictions['price'] = np.expm1(final_predictions['price'])

    final_predictions.to_csv(output_csv_path, index=False)
    print(f"\nPredictions saved to {output_csv_path}")
    print(f"Total samples processed: {len(final_predictions)}")
    return final_predictions

def extract_features_from_dataloader(model, dataloader):
    """Extract features from dataloader without KNN (for test inference)."""
    all_text_embeds = []
    all_img_embeds = []
    all_text_knn = []
    all_img_knn = []
    
    model.eval()
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting features"):
            pixel_values = batch['pixel_values'].to(next(model.parameters()).device)
            input_ids = batch['input_ids'].to(next(model.parameters()).device)
            attention_mask = batch['attention_mask'].to(next(model.parameters()).device)
            
            # Get features (will be zeros for KNN if not fitted)
            features = model(pixel_values, input_ids, attention_mask)
            
            # Convert to CPU and collect
            all_text_embeds.append(features['text_embeds'].cpu().numpy())
            all_img_embeds.append(features['img_embeds'].cpu().numpy())
            all_text_knn.append(features['text_knn'].cpu().numpy())
            all_img_knn.append(features['img_knn'].cpu().numpy())
    
    # Concatenate all features
    return {
        'text_embeds': np.concatenate(all_text_embeds, axis=0),
        'img_embeds': np.concatenate(all_img_embeds, axis=0),
        'text_knn': np.concatenate(all_text_knn, axis=0),
        'img_knn': np.concatenate(all_img_knn, axis=0)
    }

In [7]:
TEST_CSV_PATH = '../dataset/cleaned_test.csv'
TEST_IMG_DIRS = [
    '../images/test_part1',
    '../images/test_part2', 
    '../images/test_part3'
]
LGBM_MODEL_PATH = 'lgbm_price_model_final.txt'
OUTPUT_CSV_PATH = '../final_test_predictions_2.csv'

print("Test inference configuration:")
print(f"Test CSV: {TEST_CSV_PATH}")
print(f"Test image directories: {TEST_IMG_DIRS}")
print(f"LightGBM model: {LGBM_MODEL_PATH}")
print(f"Output file: {OUTPUT_CSV_PATH}")

Test inference configuration:
Test CSV: ../dataset/cleaned_test.csv
Test image directories: ['../images/test_part1', '../images/test_part2', '../images/test_part3']
LightGBM model: lgbm_price_model_final.txt
Output file: ../final_test_predictions_2.csv


In [8]:
def fit_knn_from_training_data():
    """
    Fit KNN indices using training data.
    This should be run before test inference for optimal results.
    """
    print("=" * 60)
    print("FITTING KNN INDICES FROM TRAINING DATA")
    print("=" * 60)
    
    # Load training data
    train_files = [
        '../dataset/train_split/part1.csv',
        '../dataset/train_split/part2.csv',
        '../dataset/train_split/part3.csv'
    ]
    
    train_img_dirs = [
        '../images/train_part1',
        '../images/train_part2', 
        '../images/train_part3'
    ]
    
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)
    model = CLIPPricePredictor(k_neighbors=10)
    
    # Process training data to fit KNN
    for i, (train_file, img_dir) in enumerate(zip(train_files, train_img_dirs)):
        if not os.path.exists(train_file):
            print(f"⚠️ Training file {train_file} not found, skipping...")
            continue
            
        print(f"\nFitting KNN with training data part {i+1}...")
        
        train_df = pd.read_csv(train_file)
        
        # Create training dataset
        train_dataset = PriceDataset(
            img_path=img_dir,
            image_paths=train_df['sample_id'].values,
            item_names=train_df['Item Name'].values,
            bullet_points=train_df['Bullet Points'].values,
            product_descriptions=train_df['Product Description'].values,
            values=train_df['Value'].values,
            units=train_df['Unit'].values,
            processor=processor
        )
        
        # Create dataloader with smaller batch size for memory efficiency
        train_dataloader = DataLoader(
            train_dataset,
            batch_size=8,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )
        
        # Fit KNN (this will use all training data to build indices)
        if i == 0:
            # First part: fit KNN
            model.fit_knn(train_dataloader)
        else:
            # Subsequent parts: you might want to update KNN or retrain
            # For now, we'll skip to save memory
            print(f"Skipping KNN refit for part {i+1} (using part 1 indices)")
        
        # Clean up
        del train_dataset, train_dataloader, train_df
        gc.collect()
        torch.cuda.empty_cache()
        
        # Only use first part for KNN fitting to save memory
        break
    
    print("✅ KNN indices fitted successfully!")
    
    # Save the fitted model for later use
    # Note: You might want to implement proper model saving/loading
    torch.save(model.state_dict(), 'fitted_clip_model.pth')
    
    # Also save KNN indices separately if needed
    # This would require implementing save/load methods for KNNFeatureAugmenter
    
    return model

In [9]:
def run_full_test_inference():
    """Run complete test inference pipeline."""
    try:
        predictions_df = extract_test_features_and_predict(
            test_csv_path=TEST_CSV_PATH,
            test_img_dirs=TEST_IMG_DIRS,
            trained_model_path=None,  # KNN indices not pre-loaded
            lgbm_model_path=LGBM_MODEL_PATH,
            output_csv_path=OUTPUT_CSV_PATH,
            batch_size=16  # Adjust based on GPU memory
        )
        
        print("✅ Test inference completed successfully!")
        return predictions_df
        
    except Exception as e:
        print(f"❌ Test inference failed: {e}")
        import traceback
        traceback.print_exc()
        return None


In [10]:
predictions = run_full_test_inference()

Loaded test data: 75000 samples


`torch_dtype` is deprecated! Use `dtype` instead!


Loading LightGBM model from lgbm_price_model_final.txt

Processing test part 1/3: ../images/test_part1
Extracting features for part 1...


Extracting features: 100%|██████████| 1563/1563 [00:34<00:00, 45.63it/s]


Making predictions for part 1...

Processing test part 2/3: ../images/test_part2
Extracting features for part 2...


Extracting features: 100%|██████████| 1563/1563 [00:31<00:00, 50.10it/s]


Making predictions for part 2...

Processing test part 3/3: ../images/test_part3
Extracting features for part 3...


Extracting features: 100%|██████████| 1563/1563 [00:32<00:00, 48.31it/s]


Making predictions for part 3...

Predictions saved to ../final_test_predictions_2.csv
Total samples processed: 75000
✅ Test inference completed successfully!


In [11]:
# Method 2: Use pre-extracted features (if test features are already saved)
def run_inference_with_precomputed_features():
    """Run inference using pre-extracted test features."""
    
    # Check if test features already exist
    test_feature_files = [
        'test_features_1.h5',
        'test_features_2.h5', 
        'test_features_3.h5'
    ]
    
    print("Checking for pre-extracted test features...")
    
    all_features_exist = all(os.path.exists(f) for f in test_feature_files)
    
    if not all_features_exist:
        print("❌ Pre-extracted test features not found.")
        print("Please run feature extraction first or use the full pipeline.")
        return None
    
    print("✅ Found pre-extracted test features!")
    
    # Load LightGBM model
    print(f"Loading LightGBM model from {LGBM_MODEL_PATH}")
    lgbm_model = lgb.Booster(model_file=LGBM_MODEL_PATH)
    
    # Load and combine all test features
    all_predictions = []
    all_sample_ids = []
    
    for i, feature_file in enumerate(test_feature_files, 1):
        print(f"\nProcessing test features part {i}...")
        
        # Load features
        features = load_features_from_h5(feature_file)
        
        # Prepare for LightGBM
        X_test = prepare_features_for_lgbm(features)
        
        # Make predictions
        predictions = lgbm_model.predict(X_test)
        
        # Convert from log scale back to original scale
        predictions = np.exp(predictions)
        
        # Store results (you'd need to track sample IDs separately)
        all_predictions.extend(predictions)
        
        print(f"Part {i} completed. Predictions range: {predictions.min():.2f} - {predictions.max():.2f}")
        
        # Clean up
        del features, X_test, predictions
        gc.collect()
    
    # Create output DataFrame
    # Note: You'll need to match predictions with sample IDs
    # This assumes the order is preserved
    test_df = pd.read_csv(TEST_CSV_PATH)
    
    output_df = pd.DataFrame({
        'sample_id': test_df['sample_id'].values[:len(all_predictions)],
        'predicted_price': all_predictions
    })
    
    # Save predictions
    output_df.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"\n✅ Predictions saved to {OUTPUT_CSV_PATH}")
    print(f"Total samples: {len(output_df)}")
    
    return output_df

# Uncomment to run with pre-computed features
# predictions = run_inference_with_precomputed_features()

In [12]:
# Method 3: Extract test features and save them (for later use)
def extract_and_save_test_features():
    """Extract and save test features for later inference."""
    print("=" * 60)
    print("EXTRACTING TEST FEATURES")
    print("=" * 60)
    
    # Load test data
    test_df = pd.read_csv(TEST_CSV_PATH)
    print(f"Loaded test data: {len(test_df)} samples")
    
    # Initialize components
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)
    model = CLIPPricePredictor(k_neighbors=10)
    
    # Process each test part
    for part_idx, img_dir in enumerate(TEST_IMG_DIRS):
        print(f"\nExtracting features for test part {part_idx + 1}...")
        
        # Determine samples for this part
        part_start = part_idx * (len(test_df) // len(TEST_IMG_DIRS))
        part_end = (part_idx + 1) * (len(test_df) // len(TEST_IMG_DIRS))
        if part_idx == len(TEST_IMG_DIRS) - 1:
            part_end = len(test_df)
        
        part_df = test_df.iloc[part_start:part_end].copy()
        
        if len(part_df) == 0:
            continue
        
        # Create dataset
        test_dataset = PriceDataset(
            img_path=img_dir,
            image_paths=part_df['sample_id'].values,
            item_names=part_df.get('Item Name', [None] * len(part_df)).values,
            bullet_points=part_df.get('Bullet Points', [None] * len(part_df)).values,
            product_descriptions=part_df.get('Product Description', [None] * len(part_df)).values,
            values=part_df.get('Value', [None] * len(part_df)).values,
            units=part_df.get('Unit', [None] * len(part_df)).values,
            processor=processor
        )
        
        test_dataloader = DataLoader(
            test_dataset,
            batch_size=16,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )
        
        # Extract and save features
        features = extract_features_from_dataloader(model, test_dataloader)
        
        # Add sample IDs to features
        features['sample_ids'] = part_df['sample_id'].values
        
        # Save features
        feature_file = f'test_features_{part_idx + 1}.h5'
        with h5py.File(feature_file, 'w') as f:
            f.create_dataset('text_embeds', data=features['text_embeds'], compression='gzip')
            f.create_dataset('img_embeds', data=features['img_embeds'], compression='gzip')
            f.create_dataset('text_knn', data=features['text_knn'], compression='gzip')
            f.create_dataset('img_knn', data=features['img_knn'], compression='gzip')
            f.create_dataset('sample_ids', data=features['sample_ids'].astype('S'), compression='gzip')
        
        print(f"✅ Features saved to {feature_file}")
        
        # Clean up
        del features, test_dataset, test_dataloader, part_df
        gc.collect()
        torch.cuda.empty_cache()
    
    print("\n✅ All test features extracted and saved!")

# Uncomment to extract test features
# extract_and_save_test_features()